# Sold Mortgage Rate Enrichment

In [1]:
import pandas as pd

sold = pd.read_csv("../../IDX_Data/sold_filtered_week2_3.csv")
print("Shape:", sold.shape)
sold.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_71806/3662182009.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("../../IDX_Data/sold_filtered_week2_3.csv")


Shape: (542075, 84)


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,NaN,False,NaN,NaN,90067,NaN,2105.00,177861.0,NaN,2220 Avenue Of The Stars 2704
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,0.0,False,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,NaN,16 Palisades
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,NaN,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,1.0,False,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,NaN,2250 Indian Creek Road
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,3.0,False,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,NaN,317 E. Bayfront


In [2]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url)
# Rename columns
mortgage.columns = ['date', 'rate_30yr_fixed']
# Convert to datetime
mortgage['date'] = pd.to_datetime(mortgage['date'])

mortgage.head()

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [3]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975


In [4]:
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
sold[['CloseDate', 'year_month']].head()

,CloseDate,year_month
0,NaN,NaT
1,NaN,NaT
2,NaN,NaT
3,NaN,NaT
4,NaN,NaT


In [5]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
print("Missing mortgage rates:", sold_with_rates['rate_30yr_fixed'].isnull().sum())

Missing mortgage rates: 380648


In [6]:
sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head()

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,NaN,NaT,NaN,NaN
1,NaN,NaT,NaN,NaN
2,NaN,NaT,NaN,NaN
3,NaN,NaT,NaN,NaN
4,NaN,NaT,NaN,NaN


In [7]:
sold_with_rates['rate_30yr_fixed'].describe()

count    161427.000000
mean          6.517410
std           0.327155
min           6.047500
25%           6.237500
50%           6.428000
75%           6.820000
max           7.060000
Name: rate_30yr_fixed, dtype: float64

In [8]:
sold_with_rates.to_csv("sold_with_rates.csv", index=False)
print("Sold dataset with mortgage rates saved.")

Sold dataset with mortgage rates saved.


### Merge Validation

Mortgage rate data was successfully merged into the sold dataset using a monthly key derived from CloseDate.

No missing values were found in the mortgage rate column, confirming a complete and accurate merge.